In [ ]:
pip install open_clip_torch

In [ ]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model, preprocessing function and tokenizer

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

### Prepare the Sketch-200 Dataset

In [ ]:
pip install datasets==2.16.0

In [ ]:
from datasets import load_dataset
dataset = load_dataset("songweig/imagenet_sketch")

In [ ]:
print(dataset)

In [ ]:
if sys.modules['imagenet_r_classes']:
  del sys.modules['imagenet_r_classes']

In [ ]:
from imagenet_r_classes import r_class_names, r_wnids, wnid_to_r_index

In [ ]:
import json
from torchvision.datasets.utils import download_url

# Download the official ImageNet class index mapping
download_url(
    "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json",
    "./",
    "imagenet_class_index.json",
)

# Load the JSON mapping file
with open("./imagenet_class_index.json", "r") as f:
    class_idx = json.load(f)

# Convert to a list where index 0-999 corresponds to model outputs
class_names = [class_idx[str(i)][1] for i in range(1000)]

In [ ]:
wnid_to_idx = {class_idx[str(i)][0]: i for i in range(1000)}
idx_to_wnid = {i: class_idx[str(i)][0] for i in range(1000)}

In [ ]:
r_idx_1000 = [wnid_to_idx[w] for w in r_wnids]
idx_to_r_index = {wnid_to_idx[w]: i for w, i in wnid_to_r_index.items()}

In [ ]:
from datasets import ClassLabel

keep = set(r_idx_1000)
sk200 = dataset.filter(lambda y: y in keep, input_columns="label")

In [ ]:
print(sk200)

In [ ]:
new_features = sk200['train'].features.copy()
new_features["label"] = ClassLabel(names=r_class_names)

sk200 = sk200.map(
    lambda y: {"label": idx_to_r_index[y]},
    input_columns="label",
    features=new_features,
)

In [ ]:
sk200['train'][0]

### Prepare the few shot and test data loader

In [ ]:
import collections
import random

random.seed(42)

label_to_indices = collections.defaultdict(list)
for i, y in enumerate(sk200['train']["label"]):
    label_to_indices[y].append(i)

cache_indices = []
test_indices = []

for label, indices in label_to_indices.items():
    shuffled = indices[:]
    random.shuffle(shuffled)
    cache_indices.extend(shuffled[:16])
    test_indices.extend(shuffled[16:])

print(len(cache_indices), len(test_indices))

In [ ]:
print(len(cache_indices), len(test_indices))

In [ ]:
# sk200_cache = sk200['train'].select(cache_indices)
# sk200_test = sk200['train'].select(test_indices)

In [ ]:
class HFImageDataset(Dataset):
    def __init__(self, hf_dataset, preprocess, wnid_to_index):
        self.hf_dataset = hf_dataset
        self.preprocess = preprocess
        self.wnid_to_index = wnid_to_index

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        example = self.hf_dataset[idx]
        image = self.preprocess(example["image"].convert("RGB"))
        label = example["label"]
        return image, label

In [ ]:
full_wrapped = HFImageDataset(sk200['train'], preprocess, wnid_to_r_index)

In [ ]:
from torch.utils.data import Subset

raw_few_shot_ds = Subset(full_wrapped, cache_indices)
raw_eval_ds = Subset(full_wrapped, test_indices)

print(len(raw_few_shot_ds))
print(len(raw_eval_ds))

In [ ]:
raw_few_shot_loader = DataLoader(raw_few_shot_ds, batch_size = 32, shuffle = True)
raw_eval_loader = DataLoader(raw_eval_ds, batch_size = 32)

### Build the image and text features for zero shot

In [ ]:
if sys.modules['clip_zeroshot']:
  del sys.modules['clip_zeroshot']

In [ ]:
from clip_zeroshot import load_cached_text_features, build_and_cache_image_features

In [ ]:
text_feature_cache = load_cached_text_features('/content/features/r_text-features.pt')
text_features = text_feature_cache['text_features']

In [ ]:
test_image_features_cache = build_and_cache_image_features(model, device, raw_eval_loader, './features', 'sk200_image_features')

In [ ]:
test_img_features = test_image_features_cache['image_features']
test_img_labels = test_image_features_cache['labels']

### Build the cache model for Tip-Adapter

In [ ]:
few_shot_image_cache = build_and_cache_image_features(model, device, raw_few_shot_loader, './features', 'few_shot_image_features')

In [ ]:
few_shot_image_features = few_shot_image_cache['image_features']
few_shot_image_labels = few_shot_image_cache['labels']

In [ ]:
import torch.nn.functional as F

one_hot = F.one_hot(few_shot_image_labels, num_classes=len(r_class_names))

In [ ]:
cache_keys = few_shot_image_features
cache_values = one_hot.float()

### run zero shot

In [ ]:
if sys.modules['harness']:
  del sys.modules['harness']

In [ ]:
from harness import run_comparison, zero_shot_logits, tip_adapter_logits, ece, accuracy, signed_gap

In [ ]:
metrics = {"accuracy": accuracy, "ece": ece, "signed_gap": signed_gap}

In [ ]:
shared = {
    "test_features": test_img_features.to(device),
    "labels": test_img_labels.to(device),
    "text_features": text_features.to(device),
    "cache_keys": cache_keys.to(device),
    "cache_values": cache_values.to(device),
    "logit_scale": model.logit_scale.exp()
}

In [ ]:
methods = {
    "zero_shot":   {"fn": zero_shot_logits,   "params": {}},
    "tip_adapter": {"fn": tip_adapter_logits, "params": {"alpha": 1.5, "beta": 5.0}}
}

In [ ]:
results = run_comparison(shared, methods, metrics)

In [ ]:
print(results)